# 04 会議録の全発言を BERT でベクトルにする(Google Colab で実行)

**目的**: 前処理した全発言を、日本語 BERT(`tohoku-nlp/bert-base-japanese-v3`)で 768 次元のベクトルにする。GPU が必要なので、Google Colab(GPU: Tesla T4)で実行した(PC では動かない)。

**方法**
1. 発言を 512 トークン以内のかたまりに分ける(03 で確かめた方法。Colab では Drive に置いた `text_chunk.py` から読み込む)
2. かたまりごとに BERT に入れ、最後の層のベクトルを平均する(mean pooling。前後の記号 CLS・SEP も含めて平均)
3. 1つの発言のかたまりを、長さ(トークン数)で重みを付けて平均し、発言のベクトルにする
4. 途中で止まっても続きからできるように、1万かたまりずつ保存する

**結果(先に)**: 88,297件 → 106,789かたまり → (88,297, 768) のベクトル。欠けた値(NaN)は無い。計算は約6分。

**使い道**: `05_diagnose_embeddings.ipynb` で、このベクトルで党の立場を測れるかを調べた(結論: 測れない)。05 では、その後の前処理で除いた発言を外し、88,015件を使っている。

**流れ**: 準備 → 1. かたまりに分ける → 2. かたまりを埋め込む → 3. 発言のベクトルにまとめる

## 準備

Google ドライブをつなぎ、BERT の日本語の単語分けに使う辞書(fugashi・unidic-lite)を入れる。

In [1]:
from google.colab import drive
drive.mount("/content/drive")
!pip install -q fugashi unidic-lite

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


設定:
- `TEST_N`: 数を指定すると、その件数だけで試す(`None` なら全件)
- `BATCH`: 一度に GPU に入れるかたまりの数 / `SHARD`: 何かたまりごとにファイルに保存するか

In [2]:
import sys, time, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel

BASE = Path("/content/drive/MyDrive/survey_on_party_systems")
sys.path.insert(0, str(BASE))
from text_chunk import chunk_ids

MODEL = "tohoku-nlp/bert-base-japanese-v3"
TEST_N = None
BATCH = 64
SHARD = 10000
OUT = BASE / ("emb_test" if TEST_N else "emb")
OUT.mkdir(exist_ok = True)

print(torch.cuda.get_device_name(0))

Tesla T4


## 1. かたまりに分ける

全発言をかたまりに分け、どの発言のかたまりかを記録する。かたまりを**長さの順に並べる**(`order`)のは、同じくらいの長さのかたまりを同じバッチにすると、長さをそろえるための詰め物(PAD)が少なくなり、速く計算できるため。

In [3]:
q = pd.read_parquet(
    BASE / "speeches.parquet", columns=["speechID", "text"]
)
if TEST_N:
    q = q.sample(TEST_N, random_state=0).reset_index(drop=True)
tok = AutoTokenizer.from_pretrained(MODEL)

t0 = time.time()
speech_idx, chunk_list = [], []
for i, t, in enumerate(q["text"]):
    for c in chunk_ids(t, tok):
        speech_idx.append(i)
        chunk_list.append(c)
speech_idx = np.array(speech_idx)
lengths = np.array([len(c) for c in chunk_list])
order = np.argsort(lengths, kind="stable")
print(len(q), "件 →", len(chunk_list), "かたまり", f"{time.time() - t0:.0f}秒")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


88297 件 → 106789 かたまり 182秒


**結果**: 88,297件 → **106,789かたまり**(182秒)。最初の警告は Hugging Face のトークンについての注意で、結果には影響しない。

## 2. かたまりを埋め込む

`embed_batch`: 各かたまりの前後に CLS・SEP を付け、長さをそろえて BERT に入れる。半精度(float16)で計算して速くする。最後の層のベクトルを、詰め物を除いて平均する(mean pooling)。1万かたまりごとに保存し、保存済みのものは飛ばす(途中で止まっても続きから)。

In [4]:
model = AutoModel.from_pretrained(MODEL).to("cuda").eval()
CLS, SEP, PAD = tok.cls_token_id, tok.sep_token_id, tok.pad_token_id

@torch.no_grad()
def embed_batch(batch):
    L = max(len(c) for c in batch) + 2
    ids = torch.full((len(batch), L), PAD, dtype=torch.long)
    mask = torch.zeros((len(batch), L), dtype=torch.long)
    for j, c in enumerate(batch):
        x = [CLS] + c + [SEP]
        ids[j, :len(x)] = torch.tensor(x)
        mask[j, :len(x)] = 1
    ids, mask = ids.to("cuda"), mask.to("cuda")
    with torch.autocast("cuda", dtype=torch.float16):
        h = model(input_ids=ids, attention_mask=mask).last_hidden_state
    m = mask.unsqueeze(-1).float()
    return ((h.float() * m).sum(1) / m.sum(1)).cpu().numpy()

n_shards = math.ceil(len(order) / SHARD)
t0 = time.time()
for s in range(n_shards):
    path = OUT / f"shard_{s:03d}.npy"
    if path.exists():
        continue
    idx = order[s * SHARD:(s + 1) * SHARD]
    vecs = [embed_batch([chunk_list[k] for k in idx[b:b + BATCH]])
            for b in range(0, len(idx), BATCH)]
    np.save(path, np.concatenate(vecs).astype(np.float16))
    print(f"shard {s + 1}/{n_shards} 完了  {time.time() - t0:.0f}秒")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: tohoku-nlp/bert-base-japanese-v3
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


shard 1/11 完了  6秒
shard 2/11 完了  15秒
shard 3/11 完了  29秒
shard 4/11 完了  49秒
shard 5/11 完了  75秒
shard 6/11 完了  109秒
shard 7/11 完了  149秒
shard 8/11 完了  201秒
shard 9/11 完了  262秒
shard 10/11 完了  330秒
shard 11/11 完了  378秒


**結果**: 11個のファイルに分けて保存し、合計 378秒(約6分)。読み込みのときの `UNEXPECTED` は、BERT の事前学習用の部品(文の続きを当てる部分など)で、ベクトルを作るのには使わないので問題ない。

## 3. 発言のベクトルにまとめる

保存したかたまりのベクトルを元の順番に戻し、発言ごとに、かたまりの長さで重みを付けて平均する。

In [5]:
chunk_emb = np.zeros((len(order), 768), dtype=np.float32)
for s in range(n_shards):
    idx = order[s * SHARD:(s + 1) * SHARD]
    chunk_emb[idx] = np.load(OUT / f"shard_{s:03d}.npy")

w = lengths.astype(np.float32)
speech_emb = np.zeros((len(q), 768), dtype=np.float32)
np.add.at(speech_emb, speech_idx, chunk_emb * w[:, None])
speech_emb /= np.bincount(speech_idx, weights=w, minlength=len(q))[:, None]

np.save(OUT / "embeddings.npy", speech_emb)
q[["speechID"]].to_csv(OUT / "speech_ids.csv", index=False)
print(speech_emb.shape, "NaN あり" if np.isnan(speech_emb).any() else "NaN なし")

(88297, 768) NaN なし


**結果**: (88,297, 768) のベクトル。欠けた値(NaN)は無い。`embeddings.npy` と、どの発言かを示す `speech_ids.csv` を保存した。

## まとめ

- 前処理した全発言(この時点の 88,297件)を、日本語 BERT で 768 次元のベクトルにした
- 長い発言はかたまりに分け、長さで重みを付けて平均した(03 で確かめた方法)
- 05 で、このベクトルの主な違いは話し方と長さで、党の立場は測れないことが分かった